In [16]:
import sys
sys.path.append('/home/lytq/Spatial-Transcriptomics-Benchmark/utils')
from sdmbench import compute_ARI, compute_NMI, compute_CHAOS, compute_PAS, compute_ASW, compute_HOM, compute_COM

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

import warnings
warnings.filterwarnings('ignore')

In [17]:
pred_keys = {
    'BayesSpace': 'spatial.cluster',
    'conST': 'conST_leiden',
    'SEDR': 'SEDR',
    'Seurat': 'seurat_clusters',
    'SpaceFlow': 'pred',
    'SpaGCN': 'refined_pred',
    'STAGATE': 'STAGATE',
    'stLearn': 'X_pca_kmeans',
}

umap_keys = {
    'BayesSpace': ['UMAP1', 'UMAP2'],
    'conST': ['UMAP1', 'UMAP2'],
    'SEDR': ['UMAP1', 'UMAP2'],
    'Seurat': ['umap_1', 'umap_2'],
    'SpaceFlow': ['UMAP1', 'UMAP2'],
    'SpaGCN': ['UMAP1', 'UMAP2'],
    'STAGATE': ['UMAP1', 'UMAP2'],
    'stLearn': ['UMAP1', 'UMAP2'],
}

methods = ['BayesSpace', 'conST', 'SEDR', 'Seurat', 'SpaceFlow', 'SpaGCN', 'STAGATE', 'stLearn']

SEEDS = [42, 123, 456, 789, 2024]

dataset = 'BRCA1'

data_folder = f'../../data/{dataset}'
input_dir = f'../../Results/'
output_dir = f'../../results/paper/'
os.makedirs(output_dir, exist_ok=True)

In [18]:
for method in methods:
    pred_key = pred_keys[method]
    umap_key = umap_keys[method]

    print(f'================= Processing {method} {dataset} =================')
    for seed in SEEDS:
        print(f'================== Seed {seed} =================')
    
        file = os.path.join(input_dir, f'{seed}/{dataset}/{method}')
        out_path = os.path.join(output_dir, f'{seed}/{dataset}/{method}')
        os.makedirs(out_path, exist_ok=True)
        section_id = 'V1_Human_Breast_Cancer_Block_A_Section_1'

        file_path = os.path.join(data_folder, section_id)

        adata = sc.read_visium(file_path)
        adata.var_names_make_unique()
        metadata = pd.read_csv(file + '/cell_metadata.csv', index_col=0)
        metadata.to_csv(os.path.join(out_path, 'cell_metadata.csv'))
        gt_metadata = pd.read_csv(os.path.join(file_path, 'metadata.tsv'), sep='\t')


        # Sort metadata by index based on adata
        if method not in ['SpaceFlow', 'SpaGCN', 'stLearn']:
            metadata = metadata.loc[adata.obs.index]
        # gt_metadata = gt_metadata.loc[adata.obs.index]

        # Match adata and metadata 
        # adata = adata[adata.obs.index.isin(metadata.index)]

        umap_coords = pd.read_csv(file + '/spatial_umap_coords.csv')
        umap_coords.to_csv(os.path.join(out_path, 'spatial_umap_coords.csv'))

        low_dim = pd.read_csv(file + '/low_dim_data.csv')
        low_dim.to_csv(os.path.join(out_path, 'low_dim_data.csv'), index=False)

        metrics = pd.read_csv(file + '/metrics.csv', index_col=0)


        adata.obs['gt'] = gt_metadata['fine_annot_type'].values
        # adata.obs['gt'] = metadata['fine_annot_type'].values
        # adata.obs['gt'] = metadata['cluster.init'].values
        pred = metadata[pred_key].values
        if min(pred) == 0:
            pred += 1

        adata.obs['pred'] = pred.astype(str)
        adata.obsm['X_umap'] = umap_coords[umap_key].values
        adata = adata[~pd.isnull(adata.obs['gt'])]
        adata.obs['gt'] = adata.obs['gt'].astype(str)

        time_taken, memmory_used = metrics['Time'].values[0], metrics['Memory'].values[0]
        results = {
            'SEED': seed,
            'ARI': compute_ARI(adata, 'gt', 'pred'),
            'AMI': compute_NMI(adata, 'gt', 'pred'),
            'Homogeneity': compute_HOM(adata, 'gt', 'pred'),
            'Completeness': compute_COM(adata, 'gt', 'pred'),
            'ASW': compute_ASW(adata, 'pred'),
            'CHAOS': compute_CHAOS(adata, 'pred'),
            'PAS': compute_PAS(adata, 'pred'),
            'Time (s)': time_taken,
            'Memory (MB)': memmory_used
        }
        df_results = pd.DataFrame([results], index=[method])
        df_results.to_csv(os.path.join(out_path, 'metrics.csv'), index=True)

        print(f'    Results saved to {out_path}')
        print(f'================= Finished {method} {dataset} =================')

================= Processing BayesSpace BRCA1 =================
================== Seed 42 =================
    Results saved to ../../results/paper/42/BRCA1/BayesSpace
================= Finished BayesSpace BRCA1 =================
================== Seed 123 =================
    Results saved to ../../results/paper/123/BRCA1/BayesSpace
================= Finished BayesSpace BRCA1 =================
================== Seed 456 =================
    Results saved to ../../results/paper/456/BRCA1/BayesSpace
================= Finished BayesSpace BRCA1 =================
================== Seed 789 =================
    Results saved to ../../results/paper/789/BRCA1/BayesSpace
================= Finished BayesSpace BRCA1 =================
================== Seed 2024 =================
    Results saved to ../../results/paper/2024/BRCA1/BayesSpace
================= Finished BayesSpace BRCA1 =================
================= Processing conST BRCA1 =================
================== Seed 42 

## Combine all metrics

In [38]:
all_metrics = []
output_path = f'../../results/paper/brca1_metrics.csv'

for seed in SEEDS:
    print(seed)
    input_dir = f'../../results/paper/{seed}/{dataset}'
    input_files = glob.glob(input_dir + '/*')
    input_files = [f for f in input_files if os.path.isdir(f)]

    for file in input_files:
        df_metrics = pd.read_csv(os.path.join(file, 'metrics.csv'))
        df_metrics = df_metrics.rename(columns={'Unnamed: 0': 'Method'})
        all_metrics.append(df_metrics)

all_metrics = sorted(
    all_metrics,
    key=lambda x: (x['Method'].iloc[0], x['SEED'].iloc[0])
)

with open(output_path, 'w') as f:
    for df in all_metrics:
        df.to_csv(f, index=False, header=not f.tell())

42
123
456
789
2024
